# CT explorer · From projections to a slice

Choose **Runtime → Run all** in Colab, then use the two controls below.

We will study two small inserts through a computed tomography (CT) scan. First change the **view angle** and trace what the detector sees. Then change the **number of views** and compare two reconstructions.

In [ ]:
#@title Setup and CT model
from functools import lru_cache
from io import BytesIO

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()

# The same three disks as in the lecture: x, y, radius (mm), added mu (1/mm).
# larger total volume, larger/more att. disk, smaller/less att. disk
DISKS = [(0, 0, 52, 0.019), (-13, 17, 9, 0.021), (-13, -17, 7, 0.010)]
STEP = 0.5  # mm
S = np.arange(-75, 75 + STEP, STEP)
X, Y = np.meshgrid(S, S)
ANGLES = np.arange(180)
THETA = np.deg2rad(ANGLES)
OBJECT = sum(mu * ((X-x)**2 + (Y-y)**2 <= radius**2)
             for x, y, radius, mu in DISKS)

# A parallel ray through a disk integrates mu over the length of its chord.
PROJECTIONS = sum(
    2 * mu * np.sqrt(np.maximum(
        radius**2 - (S[:, None] - x*np.cos(THETA) - y*np.sin(THETA))**2, 0))
    for x, y, radius, mu in DISKS
)
FFT_SIZE = 1024
padded = np.pad(PROJECTIONS, ((0, FFT_SIZE - len(S)), (0, 0)))
ramp = np.abs(np.fft.rfftfreq(FFT_SIZE, d=STEP))[:, None]
FILTERED = np.fft.irfft(
    np.fft.rfft(padded, axis=0) * ramp, n=FFT_SIZE, axis=0
)[:len(S)]
REFERENCE = (X-19)**2 + Y**2 <= 6**2


@lru_cache(maxsize=4)
def reconstruct(n_views):
    """Return both reconstructions using the same evenly spaced views."""
    unfiltered, filtered = np.zeros_like(X), np.zeros_like(X)
    for angle in ANGLES[::180 // n_views]:
        ray_s = X*np.cos(THETA[angle]) + Y*np.sin(THETA[angle])
        unfiltered += np.interp(ray_s, S, PROJECTIONS[:, angle], left=0, right=0)
        filtered += np.interp(ray_s, S, FILTERED[:, angle], left=0, right=0)
    # Unfiltered backprojection is scaled for a visual comparison only.
    unfiltered *= DISKS[0][3] / unfiltered[REFERENCE].mean()
    filtered *= np.pi / n_views
    return unfiltered, filtered

## What a projection measures

For one ray, $I_0$ is the incident signal and $I$ is the transmitted signal. In the single-energy model, the negative log gives the total attenuation along the ray:

$$
p=-\ln(I/I_0)=\int_{\text{ray}}\mu(x,y)\,d\ell.
$$

The local attenuation coefficient $\mu$ is in $\mathrm{mm}^{-1}$. The ray measurement $p$ is the negative log of the ratio of transmitted to incident intensity and is dimensionless. One view gives a profile of these measurements across the detector. A **sinogram** stacks the profiles by view angle. Changing the view angle rotates the rays and the detector coordinate together around the slice. Here $d\ell$ is a short distance along a ray. 

**Backprojection** spreads each measurement along its ray. **Filtered backprojection** first filters each detector profile, then spreads those values along the rays.

In [ ]:
#@title Launch the CT explorer
BLUE, TEAL, GOLD = "#173F58", "#157A80", "#A86914"


def draw_slice(ax, data, title):
    ax.imshow(data, cmap="gray", origin="lower", vmin=0, vmax=0.041,
              extent=(-75.25, 75.25, -75.25, 75.25), interpolation="nearest")
    ax.set(xlim=(-63, 63), ylim=(-63, 63), title=title)
    ax.set_axis_off()


def make_dashboard(angle, n_views):
    unfiltered, filtered = reconstruct(n_views)
    fig, axes = plt.subplots(2, 3, figsize=(10.2, 6.0), layout="constrained")
    draw_slice(axes[0, 0], OBJECT, f"Ray paths at {angle}°")
    t = np.linspace(-90, 90, 2)
    co, si = np.cos(THETA[angle]), np.sin(THETA[angle])
    # Draw rays through the two insert centers; the rays coincide at 0°.
    centers = sorted({round(x*co + y*si, 6) for x, y, _, _ in DISKS[1:]})
    for s in centers:
        axes[0, 0].plot(s*co - t*si, s*si + t*co, color="#ffca65", lw=1.4)

    profile = axes[0, 1]
    profile.plot(S, PROJECTIONS[:, angle], color=TEAL, lw=2)
    for s in centers:
        profile.axvline(s, color=GOLD, lw=1.2, linestyle="--")
    profile.set(title="One detector profile", xlabel="Detector position (mm)",
                ylabel="Line integral", xlim=(-62, 62), ylim=(0, 3))
    profile.grid(alpha=0.18)

    sinogram = axes[0, 2]
    sinogram.imshow(PROJECTIONS, extent=(-.5, 179.5, -75.25, 75.25), origin="lower",
                    aspect="auto", cmap="magma", vmin=0, vmax=3)
    sinogram.axvline(angle, color="#71f0f6", lw=1.4)
    sinogram.set(title="Sinogram · all 180 views", xlabel="View angle (°)",
                 ylabel="Detector position (mm)", xticks=[0, 45, 90, 135, 179],
                 ylim=(-62, 62))

    draw_slice(axes[1, 0], OBJECT, "Known slice")
    draw_slice(axes[1, 1], unfiltered, "Backprojection")
    draw_slice(axes[1, 2], filtered, "Filtered backprojection")
    return fig


angle_control = widgets.IntSlider(
    value=0, min=0, max=179, step=1, description="View angle (°)",
    continuous_update=False, style={"description_width": "initial"},
    layout=widgets.Layout(width="360px"))
view_control = widgets.Dropdown(
    options=[6, 12, 30, 180], value=180, description="Reconstruction views",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="265px"))
readout = widgets.HTML()
image = widgets.Image(format="png", layout=widgets.Layout(width="100%", max_width="1020px"))


def refresh(change=None):
    angle, n_views = angle_control.value, view_control.value
    fig = make_dashboard(angle, n_views)
    try:
        buffer = BytesIO()
        fig.savefig(buffer, format="png", dpi=115, facecolor="white")
        image.value = buffer.getvalue()
    finally:
        plt.close(fig)
    readout.value = (
        f"<b>Top:</b> inspect view {angle}° from the full scan. "
        f"<b>Bottom:</b> both reconstructions use {n_views} views, "
        f"spaced {180/n_views:g}° apart."
    )


angle_control.observe(refresh, names="value")
view_control.observe(refresh, names="value")
refresh()
display(widgets.VBox([
    widgets.Box([angle_control, view_control], layout=widgets.Layout(flex_flow="row wrap")),
    readout, image
]))

## What is filtered backprojection doing?

The model is a **2D slice** with three disks: a background disk and two higher-attenuation inserts. The X-rays travel **in the plane of the slice**. The scanner rotates its view within that plane, which is perpendicular to the rotation axis. Changing the view gives another projection of the same slice.

Note that a cut through the center of a uniform sphere gives the same disk in every orientation. For several spheres at different locations, tilting the slice can move it away from an insert’s center or miss that insert entirely.

### 1. Filter a detector profile at a $\theta$

The projection is $p(s,\theta)$, where $s$ is position across the detector in mm, perpendicular to the rays, and $\theta$ is the angle of that detector coordinate measured from the $x$ axis. For a fixed $\theta$, we take a **one-dimensional Fourier transform along $s$**. Its frequency variable $u$ is in cycles/mm.

$$
P(u,\theta)=\mathcal F_s\{p(s,\theta)\},
\qquad
q(s,\theta)=\mathcal F_s^{-1}\{|u|P(u,\theta)\}.
$$

The **ramp filter** $|u|$ gives higher spatial frequencies more weight. The filtered profile $q$ can have negative values. Those values help cancel the broad blur that plain backprojection leaves around features. In the widget’s sinogram, each **column** is filtered independently along detector position. In summary, we Fourier transform the profile, then filter it in frequency space, then inverse Fourier transform it. 

### 2. Backproject and add the filtered views

For a pixel at $(x,y)$, find the ray through it in each view. Its detector position is $s=x\cos\theta+y\sin\theta$. Read the filtered profile there and add the contributions:

$$
\hat\mu(x,y)\approx \Delta\theta \sum_{j=1}^{N}
q\!\left(x\cos\theta_j+y\sin\theta_j,\theta_j\right),
\qquad \Delta\theta=\pi/N.
$$

The angles are equally spaced over $180^\circ$, and $\Delta\theta$ is in radians. The code interpolates between detector samples when needed. The reconstructed pixel estimates the local linear attenuation coefficient in $\mathrm{mm}^{-1}$.

Use **Example angle θ** to choose which detector profile to filter and backproject. The left image shows what that view adds to the reconstructed slice. Its colors are signed contributions in $\mathrm{mm}^{-1}$, and $x$ and $y$ are positions in mm from the rotation center. The sum of all 180 views stays fixed as you change the example angle.

In [ ]:
#@title Explore the intermediate reconstruction steps
frequency = np.fft.rfftfreq(FFT_SIZE, d=STEP)
contribution_limit = np.abs(FILTERED).max() * np.pi / len(ANGLES)
filtered_limits = (FILTERED.min() * 1.05, FILTERED.max() * 1.05)
example_angle_control = widgets.IntSlider(
    value=90, min=0, max=179, description="Example angle θ (°)",
    continuous_update=False, style={"description_width": "initial"},
    layout=widgets.Layout(width="400px", max_width="100%"))
profile_steps = widgets.Image(format="png", layout=widgets.Layout(width="100%", max_width="1020px"))
reconstruction_steps = widgets.Image(format="png", layout=widgets.Layout(width="100%", max_width="1020px"))


def save_example_plot(fig, target):
    try:
        buffer = BytesIO()
        fig.savefig(buffer, format="png", dpi=115, facecolor="white")
        target.value = buffer.getvalue()
    finally:
        plt.close(fig)


def update_fbp_example(change=None):
    example_angle = example_angle_control.value
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.3), layout="constrained")
    axes[0].plot(S, PROJECTIONS[:, example_angle], color=TEAL)
    axes[0].set(title=f"1. Detector profile at {example_angle}°", xlabel="Detector position (mm)",
                ylabel="Line integral")
    axes[1].plot(frequency, ramp[:, 0], color=GOLD)
    axes[1].set(title="2. Ramp filter", xlabel="Frequency magnitude (cycles/mm)",
                ylabel="Weight (1/mm)")
    axes[2].plot(S, FILTERED[:, example_angle], color=BLUE)
    axes[0].set_ylim(0, PROJECTIONS.max() * 1.05)
    axes[2].set_ylim(*filtered_limits)
    axes[2].axhline(0, color="0.6", linewidth=0.8)
    axes[2].set(title="3. Filtered profile", xlabel="Detector position (mm)",
                ylabel="Filtered value (1/mm)")
    save_example_plot(fig, profile_steps)

    theta = THETA[example_angle]
    ray_position = X*np.cos(theta) + Y*np.sin(theta)
    single_view = np.interp(ray_position, S, FILTERED[:, example_angle],
                            left=0, right=0) * (np.pi / len(ANGLES))

    fig, axes = plt.subplots(1, 3, figsize=(10.8, 3.4), layout="constrained")
    bands = axes[0].imshow(single_view, origin="lower", cmap="RdBu_r",
                           vmin=-contribution_limit, vmax=contribution_limit, extent=(-75.25, 75.25)*2)
    axes[0].set(xlim=(-63, 63), ylim=(-63, 63),
                xticks=[-50, 0, 50], yticks=[-50, 0, 50],
                xlabel="x (mm)", ylabel="y (mm)",
                title=f"4. What the {example_angle}° view adds")
    fig.colorbar(bands, ax=axes[0], shrink=0.72, label="Contribution (1/mm)")
    draw_slice(axes[1], reconstruct(180)[1], "5. Sum of 180 views")
    draw_slice(axes[2], OBJECT, "Known slice")
    save_example_plot(fig, reconstruction_steps)


example_angle_control.observe(update_fbp_example, names="value")
update_fbp_example()
display(widgets.VBox([example_angle_control, profile_steps, reconstruction_steps]))

## Questions to submit to Canvas

1. **One view versus another.** Compare $0^\circ$ and $90^\circ$. What position information does the second view add?
2. **Sinogram.** Sweep the view angle slowly. Follow the highlighted column in the sinogram and the dashed marks in the detector profile. Make a rough sketch of the sinogram and label the inserts. Why do the inserts move across the detector as the view rotates? Sketch a sinogram for a sample with a high attenuation insert centered at the middle.  
3. **Compare the methods.** At **180 views**, compare the two reconstructions with the known slice. Which boundaries look sharper? Now choose **12 views**. 

The top row always shows data from the full scan. The bottom row uses an evenly spaced subset selected by the **reconstruction views** control.

The model uses ideal parallel rays, one photon energy, and no scatter or noise. The detector profiles are calculated from the chord lengths through the three disks. Clinical CT uses more complex acquisition and correction methods.